In [ ]:
!uname -a && cat /etc/*release

In [3]:
!pwd
!ls -la

/content
total 20
drwxr-xr-x 1 root root 4096 Jun 23 09:16 .
drwxr-xr-x 1 root root 4096 Jun 23 08:57 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:39 .config
-rw-r--r-- 1 root root 1757 Jun 23 09:16 deviceQuery.cu
drwxr-xr-x 1 root root 4096 Jun  4 13:39 sample_data


In [1]:
!nvidia-smi

Tue Jun 23 08:07:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
device_query_str = '''
#include <stdio.h>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);

    if (deviceCount == 0) {
        printf("Nessun dispositivo CUDA trovato.\\n");
        return 1;
    }

    for (int dev = 0; dev < deviceCount; ++dev) {
        cudaDeviceProp deviceProp;
        cudaGetDeviceProperties(&deviceProp, dev);

        printf("\\n--- DEVICE %d: \\"%s\\" ---\\n", dev, deviceProp.name);
        printf("  CUDA Capability Major/Minor version:          %d.%d\\n", deviceProp.major, deviceProp.minor);
        printf("  Total amount of global memory:                %.2f GBytes\\n", (float)deviceProp.totalGlobalMem / (1024 * 1024 * 1024));
        printf("  Number of multiprocessors (SMs):              %d\\n", deviceProp.multiProcessorCount);
        printf("  Total amount of constant memory:              %lu bytes\\n", deviceProp.totalConstMem);
        printf("  Total amount of shared memory per block:      %lu bytes\\n", deviceProp.sharedMemPerBlock);
        printf("  Total number of registers available per block: %d\\n", deviceProp.regsPerBlock);
        printf("  Warp size:                                    %d\\n", deviceProp.warpSize);
        printf("  Maximum threads per multiprocessor (SM):      %d\\n", deviceProp.maxThreadsPerMultiProcessor);
        printf("  Maximum threads per block:                    %d\\n", deviceProp.maxThreadsPerBlock);
        printf("  Max dimension size of a thread block (x,y,z): %d, %d, %d\\n", deviceProp.maxThreadsDim[0], deviceProp.maxThreadsDim[1], deviceProp.maxThreadsDim[2]);
        printf("  Max dimension size of a grid size    (x,y,z): %d, %d, %d\\n", deviceProp.maxGridSize[0], deviceProp.maxGridSize[1], deviceProp.maxGridSize[2]);
    }
    return 0;
}
'''
with open('deviceQuery.cu', 'w') as f:
    f.write(device_query_str)

In [9]:
!nvcc -arch=sm_75 deviceQuery.cu -o test

In [10]:
!/content/test


--- DEVICE 0: "Tesla T4" ---
  CUDA Capability Major/Minor version:          7.5
  Total amount of global memory:                14.56 GBytes
  Number of multiprocessors (SMs):              40
  Total amount of constant memory:              65536 bytes
  Total amount of shared memory per block:      49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                    32
  Maximum threads per multiprocessor (SM):      1024
  Maximum threads per block:                    1024
  Max dimension size of a thread block (x,y,z): 1024, 1024, 64
  Max dimension size of a grid size    (x,y,z): 2147483647, 65535, 65535


In [11]:
!nvprof /content/test

==5746== NVPROF is profiling process 5746, command: /content/test

--- DEVICE 0: "Tesla T4" ---
  CUDA Capability Major/Minor version:          7.5
  Total amount of global memory:                14.56 GBytes
  Number of multiprocessors (SMs):              40
  Total amount of constant memory:              65536 bytes
  Total amount of shared memory per block:      49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                    32
  Maximum threads per multiprocessor (SM):      1024
  Maximum threads per block:                    1024
  Max dimension size of a thread block (x,y,z): 1024, 1024, 64
  Max dimension size of a grid size    (x,y,z): 2147483647, 65535, 65535
==5746== Profiling application: /content/test
==5746== Profiling result:
No kernels were profiled.
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
      API calls:   51.87%  2.3356ms       114  20.488us      89ns  1.1759ms  cuDeviceGetAtt